In [1]:
import pandas as pd
import random
from datetime import datetime, timedelta

# ==========================================
# 1. BACA DATASET DARI KAGGLE
# ==========================================
# Ganti 'healthcareTest.csv' dengan nama file CSV asli yang Anda download dari Kaggle
nama_file_csv = 'healthcareTest.csv' 

try:
    df_kaggle = pd.read_csv(nama_file_csv)
    # Di dataset Kaggle ini, ID pasien biasanya ada di kolom 'patIndex'
    # Jika namanya berbeda, silakan ganti 'patIndex' di bawah ini
    daftar_pasien = df_kaggle['patIndex'].unique().tolist()
    print(f"✅ Berhasil membaca file CSV. Ditemukan {len(daftar_pasien)} pasien unik.")
except FileNotFoundError:
    print(f"❌ File {nama_file_csv} tidak ditemukan! Pastikan file ada di folder yang sama.")
    exit()

# ==========================================
# 2. KONFIGURASI TINDAKAN MEDIS (DUMMY)
# ==========================================
# Kita buat rata-rata tiap pasien punya 1 sampai 3 tindakan medis
jumlah_record_dummy = len(daftar_pasien) * random.randint(1, 3) 

referensi_tindakan = [
    ("CPT-80053", "Laboratory", "Comprehensive Metabolic Panel", 50.00),
    ("CPT-71045", "Radiology", "Chest X-Ray, single view", 120.00),
    ("CPT-70450", "Radiology", "CT Scan, Head/Brain", 850.00),
    ("ICD10-0DTJ", "Surgery", "Appendectomy (Open)", 8500.00),
    ("CPT-93000", "Diagnostic", "Electrocardiogram (ECG)", 75.00)
]

def generate_tanggal_acak():
    start_date = datetime(2025, 1, 1)
    end_date = datetime(2026, 5, 24)
    selisih_hari = (end_date - start_date).days
    random_hari = random.randint(0, selisih_hari)
    return start_date + timedelta(days=random_hari)

# ==========================================
# 3. GENERATE DAN EXPORT KE .SQL
# ==========================================
nama_file_sql = "kaggle_medical_procedures_dummy.sql"

with open(nama_file_sql, 'w') as file:
    # Membuat DDL Tabel
    file.write("-- Membuat Tabel Medical_Procedures untuk melengkapi Dataset Kaggle\n")
    file.write("CREATE TABLE IF NOT EXISTS Medical_Procedures (\n")
    file.write("    Procedure_ID VARCHAR(20) PRIMARY KEY,\n")
    file.write("    patIndex VARCHAR(50), -- Relasi ke dataset Kaggle\n")
    file.write("    Procedure_Date DATE,\n")
    file.write("    Procedure_Code VARCHAR(20),\n")
    file.write("    Procedure_Category VARCHAR(50),\n")
    file.write("    Procedure_Description VARCHAR(255),\n")
    file.write("    Cost DECIMAL(10, 2)\n")
    file.write(");\n\n")
    
    file.write("-- Memasukkan Data Dummy (DML)\n")
    
    print(f"⏳ Sedang membuat {jumlah_record_dummy} record tindakan medis dummy...")
    
    for i in range(1, jumlah_record_dummy + 1):
        proc_id = f"TR-{1000 + i}"
        # Pilih pasien secara acak dari dataset Kaggle
        pat_id = random.choice(daftar_pasien) 
        tgl_tindakan = generate_tanggal_acak().strftime('%Y-%m-%d')
        
        tindakan = random.choice(referensi_tindakan)
        proc_code, proc_cat, proc_desc, harga_dasar = tindakan
        
        cost = round(harga_dasar * random.uniform(0.9, 1.1), 2)
        
        insert_query = (
            f"INSERT INTO Medical_Procedures "
            f"(Procedure_ID, patIndex, Procedure_Date, Procedure_Code, Procedure_Category, Procedure_Description, Cost) "
            f"VALUES ('{proc_id}', '{pat_id}', '{tgl_tindakan}', '{proc_code}', '{proc_cat}', '{proc_desc}', {cost});\n"
        )
        file.write(insert_query)

print(f"🎉 Selesai! File '{nama_file_sql}' berhasil dibuat dan siap di-import ke Database.")

✅ Berhasil membaca file CSV. Ditemukan 344 pasien unik.
⏳ Sedang membuat 1032 record tindakan medis dummy...
🎉 Selesai! File 'kaggle_medical_procedures_dummy.sql' berhasil dibuat dan siap di-import ke Database.


In [1]:
import pandas as pd
df = pd.read_csv("healthcare_10000.csv")
df

,patIndex,Name,Age,Gender,Blood Type,Date of Admission,Discharge Date,Admission Type,Hospital,Doctor,...,Billing Amount,Medical Condition,Medication,Test Results,Procedure_ID,Procedure_Date,Procedure_Code,Procedure_Category,Procedure_Description,Cost
0,1,Aaron Hayes,39,Male,AB+,2022-07-10,2022-07-16,Elective,Nunez Inc,Dr. Ricardo Zavala,...,24102.33,Diabetes,Lipitor,Normal,TR-10001,2025-10-16,ICD10-0DTJ,Surgery,Appendectomy (Open),9327.72
1,2,Doris Hall,66,Male,B+,2019-10-09,2019-10-27,Emergency,Sanchezfort General Hospital,Dr. Adam Potter,...,26330.97,Hypertension,Lipitor,Inconclusive,TR-10002,2025-12-08,CPT-71045,Radiology,"Chest X-Ray, single view",126.39
2,3,Michele Gibson,44,Female,AB-,2020-04-17,2020-05-15,Emergency,Montoya Inc,Dr. Lee Obrien,...,16374.78,Hypertension,Aspirin,Inconclusive,TR-10003,2025-05-09,CPT-71045,Radiology,"Chest X-Ray, single view",128.37
3,4,Tammy French,59,Male,AB+,2019-01-15,2019-01-27,Elective,East Steven Regional Medical,Dr. Jeffrey Holt,...,23694.08,Asthma,Penicillin,Normal,TR-10004,2025-12-20,CPT-93000,Diagnostic,Electrocardiogram (ECG),81.06
4,5,Mary Peck,18,Female,AB-,2020-06-06,2020-06-17,Elective,Ryan & Cummings Hospital,Dr. Brandi Hernandez,...,17818.47,Cancer,Aspirin,Inconclusive,TR-10005,2025-11-14,CPT-70450,Radiology,"CT Scan, Head/Brain",892.38
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,Lucas Smith,67,Male,AB+,2023-10-14,2023-10-24,Urgent,Cherylhaven General Hospital,Dr. Jenny Bray,...,40360.24,Obesity,Ibuprofen,Normal,TR-19996,2025-09-23,CPT-71045,Radiology,"Chest X-Ray, single view",128.28
9996,9997,James Bauer,36,Female,B-,2024-05-18,2024-06-10,Emergency,Gentry Inc,Dr. Cheyenne Ball,...,15921.20,Arthritis,Ibuprofen,Inconclusive,TR-19997,2025-10-25,ICD10-0DTJ,Surgery,Appendectomy (Open),8116.51
9997,9998,Sarah Mcmillan,52,Male,A-,2020-05-13,2020-05-31,Urgent,Nelson & Schaefer Clinic,Dr. Devon Hall,...,18322.30,Cancer,Paracetamol,Inconclusive,TR-19998,2025-06-14,CPT-93000,Diagnostic,Electrocardiogram (ECG),70.38
9998,9999,Debra Friedman,37,Female,AB-,2020-04-02,2020-04-09,Elective,Mahoney & Ibarra Hospital,Dr. Joshua Smith,...,18083.06,Hypertension,Paracetamol,Abnormal,TR-19999,2025-05-03,CPT-80053,Laboratory,Comprehensive Metabolic Panel,48.18
